## Recommendation System

```
Data Description:

Unique ID of each anime.
Anime title.
Anime broadcast type, such as TV, OVA, etc.
anime genre.
The number of episodes of each anime.
The average rating for each anime compared to the number of users who gave ratings.
Number of community members for each anime.


Objective:
The objective of this assignment is to implement a recommendation system using cosine similarity on an anime dataset. 
Dataset:
Use the Anime Dataset which contains information about various anime, including their titles, genres,No.of episodes and user ratings etc.

Tasks:

Data Preprocessing:

Load the dataset into a suitable data structure (e.g., pandas DataFrame).
Handle missing values, if any.
Explore the dataset to understand its structure and attributes.

Feature Extraction:

Decide on the features that will be used for computing similarity (e.g., genres, user ratings).
Convert categorical features into numerical representations if necessary.
Normalize numerical features if required.

Recommendation System:

Design a function to recommend anime based on cosine similarity.
Given a target anime, recommend a list of similar anime based on cosine similarity scores.
Experiment with different threshold values for similarity scores to adjust the recommendation list size.
Analyze the performance of the recommendation system and identify areas of improvement.

Interview Questions:
1. Can you explain the difference between user-based and item-based collaborative filtering?
2. What is collaborative filtering, and how does it work?

```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn  as sns

#loading dataset into the enviroment
dataset = pd.read_csv("anime.csv")

# Working on a copy
df = dataset.copy()

# Understanding data set
print("\n<----------INFO----------->\n")
print(df.info())

print("\n<-----------DESCRIBE ONLY NUMERICAL---------->\n")
print(df.describe())

print("\n<---------DESCRIBE ALL NUMERICAL AND CATEGORICAL--------->\n")
print(df.describe(include='all'))

print("\n<---------MISSING VALUES--------->\n")
print(df.isnull().sum())

## Exploratory Data Analysis

In [ ]:
# Extracting numerical data and categorical data from the dataset
numerical_cols = df.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
 
print(numerical_cols)
print(categorical_cols)

In [ ]:
# Filling Missing values in the dataset
df.fillna({"name": "Unknown"},inplace=True)
df.fillna({"type": "Unknown"},inplace=True)
df.fillna({"genre": "Unknown"},inplace=True)
df.replace({"episodes":"Unknown"},pd.NA,inplace=True)
df.fillna({"episodes": df["episodes"].mode()[0]}, inplace=True)
df.fillna({"rating":df["rating"].mean()},inplace=True)
df.fillna({"members": df["members"].median()},inplace=True)

In [ ]:
print(df.isnull().sum())

In [ ]:
from collections import Counter 

# Split genre into lists per row 
genre_split = df["genre"].str.split(', ')

#Flatten to a single list of all genres, ignore NAs
all_genres = [g.strip() for sublist in genre_split.dropna() for g in sublist]

#Get the top 10 most common genes
top10 = Counter(all_genres).most_common(10)
genres,counts = zip(*top10) if top10 else([],[])

#plotting
sns.barplot(x=list(genres), y= list(counts))
plt.xticks(rotation=45)
plt.show()


## Feature Extraction

In [ ]:
# Encoding categorical data
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

# TF-IDF for genres
tfidf = TfidfVectorizer(stop_words='english')
genre_matrix = tfidf.fit_transform(df["genre"])

# One-hot encode type
ohe = OneHotEncoder()
type_matrix = ohe.fit_transform(df[["type"]])

#Combine numerical 
numerical_features = df[["episodes","rating","members"]].values
scaler = StandardScaler()
numerical_scaled = scaler.fit_transform(numerical_features)

# Final feature matrix
import scipy.sparse as sp
feature_matrix = sp.hstack([genre_matrix,type_matrix,numerical_scaled])
print(feature_matrix)

## Recommendation System

In [ ]:
# Computing cosine similarity 
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(feature_matrix, feature_matrix,dense_output=False)

## Function For Recommendation Function

In [ ]:
# Build a mapping of index to anime name
indices = pd.Series(df.index, index=df["name"]).drop_duplicates()

def get_recommendations(title, n=5,threshold=0.5):
    if title not in indices:
        return f"Anime '{title}' not found in dataset."

    # Getting the index of the anime
    idx = indices[title]

    # Getting pairwise similarity scores for that anime
    sim_scores = list(enumerate(cosine_sim[idx].toarray().ravel()))

    # Sort by similarity (highest first)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    
    sim_score_filtered = [(i, score) for i, score in sim_scores if i != idx and score > threshold]
    
    # Exclude the one we are searching and including others.
    sim_scores = sim_score_filtered[1:n+1]

    # Getting indices of similar anime
    anime_indices = [i[0] for i in sim_scores]

    # Returnning the names of recommended anime
    return df[['name', 'genre', 'type', 'rating']].iloc[anime_indices]

# Example test
print("\n<----------For threshold 0.5---------->\n")
print(get_recommendations("Dragon Ball", 10, 0.5))

print("\n<----------For threshold 0.97---------->\n")
print(get_recommendations("Dragon Ball", 10, 0.97))

## Performance and improvements
```
Strength: Simple,interpretable,no user history required.

Weakness;
-Doesn't handle new anime well.
-Only based on metadata,not actual user preferences.

Improvements:
-Hybrid model
-Deep lerning embeddings.
-Use user-item interaction matrix.
```

## Interview Questions
```
Q1. Difference between User-based and Item-based Collaborative Filtering?
Ans:
User-based CF: Finds similar users (based on rating history) and recommends items liked by those similar users.
Item-based CF: Finds similar items (based on co-ratings) and recommends them if the user liked one item.
Difference: User-based focuses on relationship between users, item-based focuses on similarity between items. 

Q2.What is Collaborative Filtering and how does it work?
Ans:
Deff: A recommendation methos that makes prediction based on past user-item interactions.

Working:
i. Build a user-item matrix using ratings and interactions.
ii. Compute similarity between users (user-based) or items (item-based).
iii. Recommend items to a user based on ratings of similar users, or items similar to what they liked.

```